# Grounded Property Content Evaluation

This notebook is the primary executable deliverable. **Run All Cells** executes the complete live pipeline—ingestion / cleaning, generation, and evaluation—across all four fixtures and writes a new Inspect AI log. Supporting Python modules contain the implementation; the notebook orchestrates and presents it.

A live run requires `ANTHROPIC_API_KEY` and uses Anthropic Claude Sonnet 4.5 for the generator and grader roles.


In [ ]:
import json
import os
from pathlib import Path

from IPython.display import Code, JSON, Markdown, display
from inspect_ai import eval_async

from property_content.inspect_task import property_content_eval

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY is required. Export it in the terminal before launching Jupyter."
    )

NOTEBOOK_LOG_DIR = Path("logs/main")
NOTEBOOK_LOG_DIR.mkdir(parents=True, exist_ok=True)


def display_result(value, *, expanded=False):
    """Render structured results as JSON and skipped/error messages as text."""
    if isinstance(value, (dict, list)):
        display(JSON(data=value, expanded=expanded))
    else:
        display(Code(str(value) if value is not None else "None", language="text"))


## Run the full live pipeline

The Inspect task processes every property independently. It extracts and verifies description facts, generates all four marketing fields in one call, applies deterministic checks, grades every generated statement against its cited facts, and evaluates editorial quality. The resulting `.eval` archive is written to `logs/main`.


In [ ]:
live_logs = await eval_async(
    property_content_eval(),
    log_dir=str(NOTEBOOK_LOG_DIR),
)

if len(live_logs) != 1:
    raise RuntimeError(f"Expected one Inspect run, received {len(live_logs)}.")

live_log = live_logs[0]
if live_log.status != "success":
    raise RuntimeError(f"Inspect evaluation failed: {live_log.error}")

{
    "status": live_log.status,
    "inspect_log": live_log.location,
    "properties_evaluated": len(live_log.samples or []),
}


## Results summary

The table below reports each named evaluation result. Grounding passes only when `grounded_statement_rate == 1.0`: every generated factual statement must be fully supported by its cited facts.


In [ ]:
CURRENT_METRICS = (
    "description_fact_support_rate",
    "schema_valid",
    "structure_valid",
    "citation_valid",
    "numeric_consistency",
    "conflict_free",
    "non_repetitive",
    "grounded_statement_rate",
    "grounding_passed",
    "editorial_quality_passed",
)

live_results = []
for sample in live_log.samples or []:
    score = sample.scores["pipeline_evaluation"]
    explanation = json.loads(score.explanation)
    live_results.append(
        {
            "case_id": sample.id,
            "raw_input": json.loads(sample.input),
            "fact_catalog": score.metadata["property_facts"]["all_facts"],
            "generated_content": score.metadata["generation_result"]["content"],
            "evaluation_metrics": {
                metric: dict(score.value)[metric] for metric in CURRENT_METRICS
            },
            "description_extraction": explanation["description_extraction"],
            "deterministic_checks": explanation["deterministic"],
            "semantic_grounding": explanation["semantic_grounding"],
            "editorial_quality": explanation["editorial_quality"],
        }
    )

[
    {"case_id": result["case_id"], **result["evaluation_metrics"]}
    for result in live_results
]


## Full input, output, and evaluation

Each property is displayed end to end so a reviewer can trace the raw source through the verified fact catalog to the generated copy and its evaluation.


In [ ]:
for result in live_results:
    display(Markdown(f"### {result['case_id']}"))
    display(Markdown("#### Raw property input"))
    display_result(result["raw_input"], expanded=False)
    display(Markdown("#### Verified fact catalog"))
    display_result(result["fact_catalog"], expanded=False)
    display(Markdown("#### Generated marketing content"))
    display_result(result["generated_content"], expanded=True)
    display(Markdown("#### Evaluation metrics"))
    display_result(result["evaluation_metrics"], expanded=True)
    display(Markdown("#### Statement-level semantic grounding"))
    display_result(result["semantic_grounding"], expanded=False)
    display(Markdown("#### Editorial-quality assessment"))
    display_result(result["editorial_quality"], expanded=False)


## Inspect audit log

The previous run created a machine-auditable Inspect archive. To review its complete prompts and model events after the notebook finishes, run:

```bash
uv run inspect view start --log-dir logs/main
```

Then open `http://127.0.0.1:7575` and select the newest run.


In [ ]:
print(f"Inspect log written to: {live_log.location}")
print("Viewer command: uv run inspect view start --log-dir logs/main")


## Secondary network-free control

This control exercises the same contracts and deterministic checks with fixture-authored extraction candidates, a deterministic generator, and fake semantic decisions. It is useful for reproducibility, but it is not a substitute for the live run above.


In [ ]:
from property_content.offline_report import build_offline_summary

offline_summary = await build_offline_summary()
offline_summary["aggregate"]
